# Website Scraper (gradio)

* Build a website scraper using gradio

In [ ]:
import os
from scraper import fetch_website_contents
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [ ]:
load_dotenv(override=True)

openai = OpenAI()

system_message = """
You are a helpful, highly skilled and professional marketing guru.
You will create a company brochure based on website contents presented to you.
You are smart enough to ignore irrelevant information such as navigational content, copyright information etc.
The brochure output should be in markdown format and should not include code blocks.
"""

In [ ]:
def message_llm(company_name, web_url, model="gpt-4.1-mini"):
    website_contents = fetch_website_contents(web_url)
    prompt = f"Create a company brochure for {company_name}. Here is their landing page:\n\n{website_contents}"
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": prompt}
    ]
    stream = openai.chat.completions.create(model=model, messages=messages, stream=True)
    result = ""
    for chunk in stream:
        result += chunk.choices[0].delta.content or ""
        yield result


In [ ]:
name_input = gr.Textbox(label="Company Name", placeholder="Enter the company name here...", lines=1)

site_input = gr.Textbox(
    label="Landing Page",
    placeholder="Enter the website URL here...",
    info="e.g., https://www.example.com",
    lines=1,
)
message_output = gr.Markdown(label="Brochure Output")

view = gr.Interface(
    fn=message_llm,
    inputs=[name_input, site_input],
    outputs=[message_output],
    examples=[
            ["Hugging Face", "https://huggingface.co", "GPT"],
            ["Edward Donner", "https://edwarddonner.com", "Claude"]
        ], 
    flagging_mode="never"
)
view.launch()